# Model Architecture Trace — Routing + Deep Per-Prompt Extraction

Generates `moe_routing_trace.json` (feeds the Model Architecture tab's prompt dropdown,
schema `{"prompts": [...]}`) plus `moe_routing_trace_umap.json` (a 2D UMAP projection of
per-domain expert activation, computed as a by-product of the same 12 forward passes).

12 short, trivia/completion-style prompts — 2 per domain (`code`, `math`, `biomedical`,
`legal`, `creative_writing`, `conversational`) — chosen for a clean next-token payoff, since
the Architecture tab's tour climaxes in a next-token-prediction reveal. This notebook is no
longer a domain-specialization data source; that job belongs to
`extract_domain_specialization.ipynb`, which uses long passages instead.

Run on a Colab A100 GPU runtime.


In [ ]:
import importlib.util
import subprocess
import sys


def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


if importlib.util.find_spec("torch") is None:
    pip_install("torch")

# transformers>=5.0 is a hard requirement: OLMoE's MoE block (OlmoeSparseMoeBlock /
# OlmoeTopKRouter / OlmoeExperts) was refactored in v5, and the hook code below targets
# that structure. Upper-bounded in case a future v6 refactors it again.
pip_install("transformers>=5.0.0,<6.0.0", "accelerate", "umap-learn", "numpy", "scikit-learn")

print("Dependency installation complete.")


In [ ]:
import json
import os
from collections import defaultdict

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "allenai/OLMoE-1B-7B-0924"
OUT_PATH = "moe_routing_trace.json"
UMAP_OUT_PATH = "moe_routing_trace_umap.json"
TOP_K_NEXT_TOKEN = 50  # candidates saved for the (later) dynamic-sampling phase

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# attn_implementation="eager": sdpa can't return attention weights, and this notebook's
# deep per-prompt trace needs them for the attention-head diagram.
model, loading_info = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
    output_loading_info=True,
)
model.eval()

assert not loading_info["missing_keys"], (
    f"Some model weights were NOT loaded from the checkpoint (randomly initialized "
    f"instead): {loading_info['missing_keys']}"
)
print(f"unexpected_keys (informational): {loading_info.get('unexpected_keys', [])}")

config = model.config
assert config.num_hidden_layers == 16, f"Expected 16 layers, got {config.num_hidden_layers}"
assert config.num_experts == 64, f"Expected 64 experts, got {config.num_experts}"
assert config.num_experts_per_tok == 8, f"Expected top-8 routing, got {config.num_experts_per_tok}"

num_heads = config.num_attention_heads
num_experts = config.num_experts
top_k_experts = config.num_experts_per_tok
hidden_size = config.hidden_size
intermediate_size = config.intermediate_size

print(f"Loaded {MODEL_ID}: {config.num_hidden_layers} layers, {num_experts} experts, top-{top_k_experts} routing")


In [ ]:
def to_float(t):
    return t.detach().float().cpu()


def downsample_1d(vec, buckets):
    n = vec.shape[0]
    idx = torch.linspace(0, n, buckets + 1).round().long()
    return [round(vec[idx[i] : max(idx[i] + 1, idx[i + 1])].mean().item(), 5) for i in range(buckets)]


def downsample_2d(mat, rows, cols):
    R, C = mat.shape
    ridx = torch.linspace(0, R, rows + 1).round().long()
    cidx = torch.linspace(0, C, cols + 1).round().long()
    out = []
    for i in range(rows):
        r0, r1 = ridx[i].item(), max(ridx[i].item() + 1, ridx[i + 1].item())
        row_vals = []
        for j in range(cols):
            c0, c1 = cidx[j].item(), max(cidx[j].item() + 1, cidx[j + 1].item())
            row_vals.append(round(mat[r0:r1, c0:c1].mean().item(), 5))
        out.append(row_vals)
    return out


# downsample resolutions -- kept modest since all 12 prompts get the full heavy trace
ROUTER_GRID = (10, 12)   # downsample of W_router [64, 2048] for the diagram
HIDDEN_STRIP = 20        # downsample length of hidden vectors
EXPERT_GRID = (5, 5)     # downsample of each expert's gate/up/down weight matrix
ATTN_GRID = (10, 10)     # downsample of each [2048,2048] Q/K/V/O weight matrix
HEAD_STRIP = 6           # downsample width for a single head's real 128-dim Q/K/V slice


In [ ]:
# 6 domains x 2 short, trivia/completion-style prompts each = 12 total. Chosen for a clean
# next-token payoff (matching the original 5 prompts' spirit) rather than authentic
# domain-register text, since the Architecture tab's tour climaxes in a next-token-prediction
# reveal and needs a narratively satisfying completion regardless of domain.
PROMPTS = {
    "code": [
        "The most popular programming language for data science is",
        "A loop that never terminates is called an infinite",
    ],
    "math": [
        "The square root of sixteen is",
        "Two plus two equals",
    ],
    "biomedical": [
        "The organ that pumps blood throughout the human body is the",
        "White blood cells are a key part of the body's immune",
    ],
    "legal": [
        "The document that establishes the fundamental laws of the United States is called the",
        "A person accused of a crime is presumed",
    ],
    "creative_writing": [
        "It was a dark and stormy",
        "Roses are red, violets are",
    ],
    "conversational": [
        "Thank you so much, I really appreciate",
        "It was great catching up, see you",
    ],
}

domains = list(PROMPTS.keys())
total_prompts = sum(len(v) for v in PROMPTS.values())
print(f"Domains: {domains}")
print(f"Total prompts: {total_prompts}")
assert total_prompts == 12, f"Expected 12 prompts (6 x 2), got {total_prompts}"
for domain, examples in PROMPTS.items():
    assert len(examples) == 2, f"{domain} has {len(examples)} prompts, expected 2"


In [ ]:
def extract_for_prompt(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    token_ids = inputs["input_ids"][0].tolist()
    token_strs = [tokenizer.decode([tid]) for tid in token_ids]

    router_inputs = {}
    ln1_outputs, q_raw, k_raw, v_raw, q_normed, k_normed, attn_outputs = {}, {}, {}, {}, {}, {}, {}
    hooks = []

    def make_pre_hook(store, layer_idx):
        def hook(module, args):
            store[layer_idx] = args[0].detach().float().cpu()
        return hook

    def make_post_hook(store, layer_idx):
        def hook(module, args, output):
            val = output[0] if isinstance(output, tuple) else output
            store[layer_idx] = val[0].detach().float().cpu()  # drop batch dim -> [seq, hidden]
        return hook

    for li, layer in enumerate(model.model.layers):
        hooks.append(layer.mlp.gate.register_forward_pre_hook(make_pre_hook(router_inputs, li)))
        hooks.append(layer.input_layernorm.register_forward_hook(make_post_hook(ln1_outputs, li)))
        hooks.append(layer.self_attn.q_proj.register_forward_hook(make_post_hook(q_raw, li)))
        hooks.append(layer.self_attn.k_proj.register_forward_hook(make_post_hook(k_raw, li)))
        hooks.append(layer.self_attn.v_proj.register_forward_hook(make_post_hook(v_raw, li)))
        hooks.append(layer.self_attn.q_norm.register_forward_hook(make_post_hook(q_normed, li)))
        hooks.append(layer.self_attn.k_norm.register_forward_hook(make_post_hook(k_normed, li)))
        hooks.append(layer.self_attn.register_forward_hook(make_post_hook(attn_outputs, li)))

    with torch.no_grad():
        outputs = model(**inputs, output_router_logits=True, output_attentions=True, output_hidden_states=True)

    for h in hooks:
        h.remove()

    router_logits = outputs.router_logits
    num_layers = len(router_logits)

    layers_trace = []
    for layer_idx, layer_logits in enumerate(router_logits):
        probs = torch.softmax(layer_logits.float(), dim=-1)
        topk = torch.topk(probs, k=top_k_experts, dim=-1)
        tokens_trace = []
        for t in range(len(token_ids)):
            tokens_trace.append(
                {
                    "token_index": t,
                    "top_experts": topk.indices[t].tolist(),
                    "top_weights": [round(w, 5) for w in topk.values[t].tolist()],
                    "all_probs": [round(p, 5) for p in probs[t].tolist()],
                }
            )
        layers_trace.append({"layer": layer_idx, "tokens": tokens_trace})

    next_token_logits = outputs.logits[0, -1, :]
    next_token_probs = torch.softmax(next_token_logits.float(), dim=-1)
    top_next = torch.topk(next_token_probs, k=TOP_K_NEXT_TOKEN)
    next_token_candidates = [
        {"token": tokenizer.decode([tid]), "prob": round(p, 6)}
        for p, tid in zip(top_next.values.tolist(), top_next.indices.tolist())
    ]

    router_matrices = []
    hidden_vectors = []
    for li in range(num_layers):
        w = to_float(model.model.layers[li].mlp.gate.weight)
        router_matrices.append(downsample_2d(w, *ROUTER_GRID))
        h = router_inputs[li]
        hidden_vectors.append([downsample_1d(h[t], HIDDEN_STRIP) for t in range(len(token_ids))])

    active_pairs = sorted(
        {(li, e) for li, lt in enumerate(layers_trace) for tt in lt["tokens"] for e in tt["top_experts"]}
    )

    expert_weights = {}
    weight_cache = {}
    for li, e in active_pairs:
        gu = to_float(model.model.layers[li].mlp.experts.gate_up_proj[e])
        dp = to_float(model.model.layers[li].mlp.experts.down_proj[e])
        gate_w = gu[:intermediate_size, :]
        up_w = gu[intermediate_size:, :]
        weight_cache[(li, e)] = (gate_w, up_w, dp)
        expert_weights[f"{li}_{e}"] = {
            "gate": downsample_2d(gate_w, *EXPERT_GRID),
            "up": downsample_2d(up_w, *EXPERT_GRID),
            "down": downsample_2d(dp, *EXPERT_GRID),
        }

    silu = torch.nn.functional.silu
    expert_outputs = {}
    for li, lt in enumerate(layers_trace):
        h_layer = router_inputs[li]
        for t, tt in enumerate(lt["tokens"]):
            h_t = h_layer[t]
            for e in tt["top_experts"]:
                gate_w, up_w, dp = weight_cache[(li, e)]
                gate_pre = torch.nn.functional.linear(h_t, gate_w)
                up = torch.nn.functional.linear(h_t, up_w)
                mid = silu(gate_pre) * up
                out = torch.nn.functional.linear(mid, dp)
                expert_outputs[f"{t}_{li}_{e}"] = downsample_1d(out, HIDDEN_STRIP)

    hidden_states_all = [h[0].detach().float().cpu() for h in outputs.hidden_states]
    embed_vec = to_float(model.model.embed_tokens.weight)[torch.tensor(token_ids)]
    embed_strip = [downsample_1d(embed_vec[t], HIDDEN_STRIP) for t in range(len(token_ids))]

    # ---- Rotary Position Embeddings (RoPE), replicated exactly from OlmoeRotaryEmbedding /
    # apply_rotary_pos_emb in modeling_olmoe.py. Real OLMoE attention is: project -> RMSNorm
    # (q_norm/k_norm, on the FULL hidden width) -> split into heads -> RoPE rotation -> Q.K^T.
    # Our hooks on q_norm/k_norm capture the PRE-RoPE tensors, so without this step the
    # "Q head"/"K head" grids shown to the user would not be the actual values the model
    # multiplies together -- this recomputes the real post-RoPE values from the real q_norm/
    # k_norm outputs already captured, using the model's own rope_theta.
    def rope_cos_sin(seq_len, head_dim, rope_theta):
        inv_freq = 1.0 / (rope_theta ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        pos = torch.arange(seq_len, dtype=torch.float32)
        freqs = torch.outer(pos, inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        return emb.cos(), emb.sin()

    def rotate_half(x):
        x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2 :]
        return torch.cat((-x2, x1), dim=-1)

    def apply_rope(x, cos, sin):
        return x * cos + rotate_half(x) * sin

    rope_theta = getattr(model.config, "rope_theta", None) or model.config.rope_parameters["rope_theta"]
    head_dim_for_rope = model.model.layers[0].self_attn.head_dim
    rope_cos, rope_sin = rope_cos_sin(len(token_ids), head_dim_for_rope, rope_theta)

    per_layer_flow = []
    attn = None
    for li in range(num_layers):
        layer = model.model.layers[li]
        attn = layer.self_attn

        q_w = downsample_2d(to_float(attn.q_proj.weight), *ATTN_GRID)
        k_w = downsample_2d(to_float(attn.k_proj.weight), *ATTN_GRID)
        v_w = downsample_2d(to_float(attn.v_proj.weight), *ATTN_GRID)
        o_w = downsample_2d(to_float(attn.o_proj.weight), *ATTN_GRID)

        layer_in = hidden_states_all[li]
        layer_out = hidden_states_all[li + 1]
        attn_out = attn_outputs[li]
        after_attn_residual = layer_in + attn_out
        moe_out = layer_out - after_attn_residual

        attn_probs_all_heads = [
            [[round(v, 5) for v in row] for row in outputs.attentions[li][0, head].detach().float().cpu().tolist()]
            for head in range(num_heads)
        ]

        hd = attn.head_dim
        q_by_head, k_by_head, v_by_head, head_output_by_head = [], [], [], []
        q_by_head_prerope, k_by_head_prerope = [], []
        for h in range(num_heads):
            v_head_full = v_raw[li][:, h * hd : (h + 1) * hd]
            q_head_prerope = q_normed[li][:, h * hd : (h + 1) * hd]
            k_head_prerope = k_normed[li][:, h * hd : (h + 1) * hd]
            q_head_rope = apply_rope(q_head_prerope, rope_cos, rope_sin)
            k_head_rope = apply_rope(k_head_prerope, rope_cos, rope_sin)
            q_by_head_prerope.append([downsample_1d(q_head_prerope[t], HEAD_STRIP) for t in range(len(token_ids))])
            k_by_head_prerope.append([downsample_1d(k_head_prerope[t], HEAD_STRIP) for t in range(len(token_ids))])
            q_by_head.append([downsample_1d(q_head_rope[t], HEAD_STRIP) for t in range(len(token_ids))])
            k_by_head.append([downsample_1d(k_head_rope[t], HEAD_STRIP) for t in range(len(token_ids))])
            v_by_head.append([downsample_1d(v_head_full[t], HEAD_STRIP) for t in range(len(token_ids))])
            head_out_full = outputs.attentions[li][0, h].detach().float().cpu() @ v_head_full
            head_output_by_head.append([downsample_1d(head_out_full[t], HEAD_STRIP) for t in range(len(token_ids))])

        per_layer_flow.append(
            {
                "ln1_weight": downsample_1d(to_float(layer.input_layernorm.weight), HIDDEN_STRIP),
                "ln1_out": [downsample_1d(ln1_outputs[li][t], HIDDEN_STRIP) for t in range(len(token_ids))],
                "q_weight": q_w, "k_weight": k_w, "v_weight": v_w, "o_weight": o_w,
                "q_raw": [downsample_1d(q_raw[li][t], HIDDEN_STRIP) for t in range(len(token_ids))],
                "k_raw": [downsample_1d(k_raw[li][t], HIDDEN_STRIP) for t in range(len(token_ids))],
                "v_raw": [downsample_1d(v_raw[li][t], HIDDEN_STRIP) for t in range(len(token_ids))],
                "q_normed": [downsample_1d(q_normed[li][t], HIDDEN_STRIP) for t in range(len(token_ids))],
                "k_normed": [downsample_1d(k_normed[li][t], HIDDEN_STRIP) for t in range(len(token_ids))],
                "attn_output": [downsample_1d(attn_out[t], HIDDEN_STRIP) for t in range(len(token_ids))],
                "attn_probs_all_heads": attn_probs_all_heads,
                "q_by_head": q_by_head, "k_by_head": k_by_head, "v_by_head": v_by_head,
                "q_by_head_prerope": q_by_head_prerope, "k_by_head_prerope": k_by_head_prerope,
                "head_output_by_head": head_output_by_head,
                "after_attn_residual": [downsample_1d(after_attn_residual[t], HIDDEN_STRIP) for t in range(len(token_ids))],
                "ln2_weight": downsample_1d(to_float(layer.post_attention_layernorm.weight), HIDDEN_STRIP),
                "ln2_out": [downsample_1d(router_inputs[li][t], HIDDEN_STRIP) for t in range(len(token_ids))],
                "moe_output": [downsample_1d(moe_out[t], HIDDEN_STRIP) for t in range(len(token_ids))],
                "layer_output": [downsample_1d(layer_out[t], HIDDEN_STRIP) for t in range(len(token_ids))],
            }
        )

    layer_flow = {
        "num_attention_heads": num_heads,
        "head_dim": attn.head_dim,
        "embed_strip": embed_strip,
        "per_layer": per_layer_flow,
        "grid_dims": {"attn": list(ATTN_GRID)},
    }

    trace = {
        "prompt": prompt,
        "model_id": MODEL_ID,
        "num_layers": num_layers,
        "num_experts": num_experts,
        "top_k_experts": top_k_experts,
        "hidden_size": hidden_size,
        "intermediate_size": intermediate_size,
        "tokens": [{"index": i, "text": s} for i, s in enumerate(token_strs)],
        "layers": layers_trace,
        "next_token_candidates": next_token_candidates,
        "router_matrices": router_matrices,
        "hidden_vectors": hidden_vectors,
        "expert_weights": expert_weights,
        "expert_outputs": expert_outputs,
        "grid_dims": {"router": list(ROUTER_GRID), "hidden_strip": HIDDEN_STRIP, "expert": list(EXPERT_GRID)},
        "layer_flow": layer_flow,
    }
    return trace, len(active_pairs)


In [ ]:
all_traces = []
for domain, prompts in PROMPTS.items():
    for prompt in prompts:
        print(f"[{len(all_traces) + 1}/{total_prompts}] ({domain}) extracting: {prompt!r}")
        trace, n_pairs = extract_for_prompt(prompt)
        trace["domain"] = domain
        all_traces.append(trace)
        print(f"    tokens={[t['text'] for t in trace['tokens']]}  top_pred={trace['next_token_candidates'][0]}  active_pairs={n_pairs}")

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump({"prompts": all_traces}, f)

print(f"\nWrote {len(all_traces)} prompts to {OUT_PATH} ({os.path.getsize(OUT_PATH) / 1e6:.2f} MB)")


## UMAP: per-domain expert activation

Reuses the top-8 routing decisions already captured in `all_traces` above (no extra
forward passes) to build one activation-rate vector per (layer, expert) pair, one
dimension per domain, and projects it to 2D with the same cosine-metric UMAP method as
`extract_domain_specialization.ipynb`. Signal here is thinner (12 short prompts vs. 6 long
passages), but the data exists as a free by-product if we want to surface it later.


In [ ]:
import umap

NUM_LAYERS = config.num_hidden_layers
NUM_EXPERTS = config.num_experts
domain_to_idx = {d: i for i, d in enumerate(domains)}

# activation_counts[layer][expert][domain_idx] = count of this domain's tokens with this
# expert in top-8 at this layer. tokens_per_domain[domain_idx] = total tokens seen for that
# domain (same at every layer, since every layer sees the same tokenized prompts).
activation_counts = np.zeros((NUM_LAYERS, NUM_EXPERTS, len(domains)), dtype=np.float64)
tokens_per_domain = np.zeros(len(domains), dtype=np.float64)

expert_token_scores = defaultdict(list)  # (layer, expert) -> [(weight, token, domain, prompt)]

for trace in all_traces:
    d_idx = domain_to_idx[trace["domain"]]
    n_tokens = len(trace["tokens"])
    tokens_per_domain[d_idx] += n_tokens
    for layer_trace in trace["layers"]:
        li = layer_trace["layer"]
        for tt in layer_trace["tokens"]:
            token_text = trace["tokens"][tt["token_index"]]["text"]
            for e, w in zip(tt["top_experts"], tt["top_weights"]):
                activation_counts[li, e, d_idx] += 1
                expert_token_scores[(li, e)].append((w, token_text, trace["domain"], trace["prompt"]))

tokens_per_domain[tokens_per_domain == 0] = 1  # guard divide-by-zero
activation_rates = activation_counts / tokens_per_domain[None, None, :]
expert_vectors = activation_rates.reshape(NUM_LAYERS * NUM_EXPERTS, len(domains))

point_layer_ids = np.repeat(np.arange(NUM_LAYERS), NUM_EXPERTS)
point_expert_ids = np.tile(np.arange(NUM_EXPERTS), NUM_LAYERS)

# Never-activated (layer, expert) pairs are all-zero and undefined under the cosine metric
# (0/0 -> NaN) -- exclude from the projection, report separately as excluded_experts.
active_mask = expert_vectors.sum(axis=1) > 0
active_vectors = expert_vectors[active_mask]

reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1, metric="cosine", n_jobs=1)
active_embedding = reducer.fit_transform(active_vectors)

assert not np.isnan(active_embedding).any(), (
    "UMAP produced NaN coordinates even after excluding all-zero rows -- inspect "
    "active_vectors for degenerate rows, or re-run with metric='euclidean'."
)

print(f"Built {expert_vectors.shape[0]} (layer, expert) vectors across {len(domains)} domains.")
print(f"UMAP embedding shape: {active_embedding.shape} ({int(active_mask.sum())} active of {expert_vectors.shape[0]} total pairs)")

TOP_K_TOKENS = 8
active_indices = np.flatnonzero(active_mask)

umap_points = []
for row, i in enumerate(active_indices):
    layer_id = int(point_layer_ids[i])
    expert_id = int(point_expert_ids[i])
    vec = expert_vectors[i]
    dominant_domain = domains[int(np.argmax(vec))]

    samples = sorted(expert_token_scores.get((layer_id, expert_id), []), key=lambda item: -item[0])[:TOP_K_TOKENS]
    top_tokens = [
        {"token": tok, "score": round(float(score), 4), "domain": dom, "prompt": prompt}
        for score, tok, dom, prompt in samples
    ]

    umap_points.append({
        "layer_id": layer_id,
        "expert_id": expert_id,
        "x": round(float(active_embedding[row, 0]), 4),
        "y": round(float(active_embedding[row, 1]), 4),
        "dominant_domain": dominant_domain,
        "domain_activation_rate": {d: round(float(vec[j]), 4) for j, d in enumerate(domains)},
        "top_tokens": top_tokens,
    })

excluded_experts = [
    {"layer_id": int(point_layer_ids[i]), "expert_id": int(point_expert_ids[i])}
    for i in np.flatnonzero(~active_mask)
]

assert len(umap_points) + len(excluded_experts) == NUM_LAYERS * NUM_EXPERTS

umap_data = {
    "domains": domains,
    "num_layers": NUM_LAYERS,
    "num_experts": NUM_EXPERTS,
    "points": umap_points,
    "excluded_experts": excluded_experts,
}

with open(UMAP_OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(umap_data, f, ensure_ascii=False, allow_nan=False, indent=2)

print(f"Wrote {UMAP_OUT_PATH} ({len(umap_points)} points, {len(excluded_experts)} excluded pairs)")


In [ ]:
try:
    from google.colab import files
    files.download(OUT_PATH)
    files.download(UMAP_OUT_PATH)
except ImportError:
    print("Not running in Google Colab -- skipping auto-download.")
    print(f"Files were written locally at: {OUT_PATH} and {UMAP_OUT_PATH}")
